# Notebook 03 — XAI Heatmap Validation

This notebook:
1. Generates attention rollout heatmaps for 20 test samples
2. Generates GradCAM heatmaps for comparison
3. Generates character-level attribution heatmaps
4. Runs the ROAR (Remove and Retrain) faithfulness test:
   - Masks top-20% attention pixels
   - Re-runs TrOCR on masked images
   - Measures CER increase (larger = more faithful heatmaps)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import jiwer
from PIL import Image
from pathlib import Path
import random

from src.ocr import TrOCREngine
from src.xai import XAIGenerator

print('Imports OK')

In [ ]:
# ── Load model and XAI generator ──────────────────────────────────────────────
engine = TrOCREngine(model_path='../models/trocr_finetuned')  # or default
xai = XAIGenerator(
    model=engine._model,
    processor=engine._processor,
    device=engine._device,
)
print('Model and XAI generator loaded.')

In [ ]:
# ── Visualise 20 Attention Rollout Heatmaps ───────────────────────────────────
# Load 20 test samples (from notebook 00)
test_samples = []  # ... (populate from IAM)

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for ax, s in zip(axes.flat, test_samples[:20]):
    img = Image.open(s['path'])
    rollout = xai.generate_attention_rollout(img)
    overlay = xai.generate_overlay(img, rollout)
    ax.imshow(overlay)
    ax.set_title(f'Label: {s["label"]}', fontsize=9)
    ax.axis('off')

plt.suptitle('Attention Rollout Overlays — Top 20 Test Samples', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/attention_rollout_gallery.png', dpi=80)
plt.show()
print('Check: highlighted regions should correspond to the word strokes, not background.')

In [ ]:
# ── ROAR Faithfulness Test ────────────────────────────────────────────────────
import cv2

def apply_roar_mask(image, heatmap, mask_fraction=0.20):
    """Mask top-{mask_fraction} highest-attention pixels with grey."""
    arr = np.array(image.convert('RGB'))
    threshold = np.quantile(heatmap, 1 - mask_fraction)
    mask = heatmap >= threshold  # bool array
    arr[mask] = 128  # replace with neutral grey
    return Image.fromarray(arr)

roar_preds, roar_refs, orig_preds = [], [], []

for s in test_samples[:20]:
    img = Image.open(s['path'])

    # Original prediction
    orig_cands = engine.recognise(img)
    orig_preds.append(orig_cands[0].word if orig_cands else '')

    # ROAR masked prediction
    rollout = xai.generate_attention_rollout(img)
    masked = apply_roar_mask(img, rollout, 0.20)
    roar_cands = engine.recognise(masked)
    roar_preds.append(roar_cands[0].word if roar_cands else '')

    roar_refs.append(s['label'])

orig_cer = jiwer.cer(roar_refs, orig_preds)
roar_cer = jiwer.cer(roar_refs, roar_preds)

print(f'Original CER:    {orig_cer*100:.2f}%')
print(f'ROAR-masked CER: {roar_cer*100:.2f}%')
print(f'CER increase:    {(roar_cer - orig_cer)*100:.2f} percentage points')
print('Higher CER increase = more faithful heatmaps (masking important pixels hurts OCR).')